[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/06_multihead_attention.ipynb)

# 🔴 困难：多头注意力（Multi-Head Attention）

从零实现 **Multi-Head Attention** —— Transformer 的核心构建模块。

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

### 函数签名
```python
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, Q, K, V) -> torch.Tensor: ...
```

### 要求
- 使用 `nn.Linear(d_model, d_model)` 创建 `self.W_q`、`self.W_k`、`self.W_v`、`self.W_o`
- 每个头的维度 `d_k = d_model // num_heads`
- `forward(Q, K, V)`：Q 的形状为 `(B, seq_q, d_model)`，K 和 V 的形状为 `(B, seq_k, d_model)`
- 必须支持 **交叉注意力**（cross-attention，即 `seq_q != seq_k`）
- **禁止** 使用 `torch.nn.MultiheadAttention`
- **允许** 使用 `torch.softmax` 和 `torch.matmul`

### 实现步骤
1. 线性投影：`q = self.W_q(Q)`，`k = self.W_k(K)`，`v = self.W_v(V)`
2. 重塑形状为 `(B, num_heads, seq, d_k)`
3. 对每个头执行带缩放的点积注意力
4. 拼接所有头 → `(B, seq_q, d_model)`
5. 输出投影：`self.W_o(concat)`

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn as nn
import math

/Users/xiaohui/Documents/project/TorchCode/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


1. 线性投影并重塑形状 (B, seq, d_model) -> (B, seq, h, d_k)
2. (B, seq, h, d_k) -> (B, h, seq, d_k) 方便计算，将 seq 和 head 转置
3. score = (B, h, seq_q, d_k) * (B, h, d_k, seq_k) -> (B, h, seq_q, seq_k) 注意 seq_q != seq_k
4. 如果有 mask, 根据 mask==0 的位置，将对应位置的分数设置成 -∞
5. 将 score 的最后一层归一化，也就是 seq_k 的维度
6. 点乘 v -> (B, h, seq_q, d_k) 
7. (B, h, seq_q, d_k) -> (B, seq_q, h, d_k) -> (B, seq_q, d_model)
8. 线性变换 W_o(o) (B, seq_q, d_model) * (d_model, d_model) -> (B, seq_q, d_model) 
9. 注意，在矩阵转置后，在 GPU 显存中就不连续了，需要调用 .contiguous() 后调用 .view()

Q * w_q -> q
K * w_k -> k
v * w_v -> v

O = softmax(q * k_t / sqrt(d_k)) * v
o = w_o * O

注意: 
- tensor.masked_fill(input: Tensor, mask: Tensor, value: Tensor) 表示将所有满足 func 的值都设置成 value, pytorch 惯例，方法尾部不带 _ 的一般是创建新的内存,不创建新的内存使用 tensor.masked_fill_(input: Tensor, mask: Tensor, value: Tensor)
- PyTorch 的自动求导机制（Autograd）需要保留前向传播时的某些张量状态来计算梯度。所以前向传播一般使用非原位操作
- 一般在不需要计算梯度的操作的时候使用 in-place 操作

In [ ]:
# ✏️ 在此处实现你的代码

class MultiHeadAttention(nn.Module):
    def __init__(self, 
                 d_model: int, 
                 num_heads: int):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model 必须能被 num_heads 整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # 定义线性变换层
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self,
                Q: torch.Tensor, 
                K: torch.Tensor, 
                V: torch.Tensor, 
                mask=None):
        batch_size = Q.size(0)
        ''' 这里默认 QKV 的输入 dim 等于 d_model '''
        # 1. 线性投影并重塑形状 (B, seq, d_model) -> (B, seq, h, d_k)
        # 然后转置为 (B, h, seq, d_k) 以便进行批矩阵乘法
        q = self.W_q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_k(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_v(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 2. 计算缩放点积注意力 (Scaled Dot-Product Attention)
        # scores 形状: (B, h, seq_q, seq_k)
        attn_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        torch.masked_fill
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        
        attn_weights = torch.softmax(attn_scores, dim=-1)
        
        # 3. 注意力权重应用到 V 上: (B, h, seq_q, d_k)
        output = torch.matmul(attn_weights, v)
        
        # 4. 拼接 (Concat) 所有头: (B, h, seq_q, d_k) -> (B, seq_q, h * d_k)
        # 注意: transpose 之后必须调用 contiguous() 才能使用 view
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # 5. 最后一次线性投影
        return self.W_o(output)

In [ ]:
# 🧪 调试测试
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
print("W_q 类型:", type(mha.W_q))                    # 应为 nn.Linear
print("W_q.weight 形状:", mha.W_q.weight.shape)      # (32, 32)

x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
print("输出形状:", out.shape)                        # (2, 6, 32)

# 交叉注意力测试
Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("交叉注意力形状:", out2.shape)                 # (1, 3, 32)

In [ ]:
# ✅ 提交验证
from torch_judge import check
check("mha")